# Notebook 03C — FDIC History: Structural Events & Exit Evidence Audit

**Part 2 extension, run after Notebook 03B.** This notebook adds independently downloaded FDIC BankFind `/history` events, verifies acquisition completeness, and revises **retrospective outcome references only**. It does **not** rewrite the original Financials Parquet, 03B clean panel, or 03B outcomes.

**Important:** History contains both **institutional** and **branch-level** events. A branch acquisition or bank-name change is *not* evidence that a bank exited. Classification uses the outgoing institution role (`OUT_CERT`), event type, event effective date, and an explicit temporal window. Ambiguous cases remain `REVIEW`.

Data sources: [FDIC BankFind API documentation](https://api.fdic.gov/banks/docs/) and [FDIC Bank Structure Changes](https://banks.data.fdic.gov/bankfind-suite/oscr).

## 0. Setup and output isolation

`RUN_MODE='FULL'` queries all effective-dated history events from 1984-01-01 to 2026-06-30. The downloader caches **one verified file per year** and resumes safely across Colab resets. It may take time because the History endpoint contains many branch events. For a quick API test, set `RUN_MODE='PILOT'`; **do not draw completeness conclusions from PILOT**.

In [ ]:
%pip -q install "requests>=2.31,<3" "pyarrow>=17,<25" "pandas>=2.2,<3" "numpy>=1.24,<3"

In [ ]:
import os, json, time, hashlib, shutil, tempfile, zipfile, re, platform, sys
from collections import Counter
from datetime import date, datetime, timezone, timedelta
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import pyarrow.parquet as pq

RUN_MODE = 'FULL'       # 'PILOT' tests 2025 only; FULL required for final exit classification
PAGE_LIMIT = 7000       # Each date segment fetched as ONE page to avoid unstable offset ties
TIMEOUT_SEC = 90
RETRIES = 5
SLEEP_BETWEEN_CALLS = 0.12
REFRESH_VERIFIED_HISTORY = False
STUDY_START = date(1984, 1, 1)
STUDY_END = date(2026, 6, 30)
MAX_EXIT_GAP_DAYS = 366
MAX_ENDDATE_DISAGREEMENT_DAYS = 45  # secondary consistency check vs 03B institutions ENDEFYMD

root_override = os.environ.get('FDIC_PROJECT_ROOT')
if root_override:
    PROJECT_ROOT = Path(root_override).expanduser().resolve()
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/FDIC_Bank_Failure_Project')

HISTORY_RAW_DIR = PROJECT_ROOT / 'data_raw/fdic_history_effective_1984_2026_v1'
HISTORY_MANIFEST = HISTORY_RAW_DIR / 'history_manifest.json'
PRIOR_OUTCOMES = PROJECT_ROOT / 'outputs/part2/cleaning_v1/bank_exit_outcomes_retrospective.csv'
PRIOR_SUMMARY = PROJECT_ROOT / 'outputs/part2/cleaning_v1/cleaning_summary.json'
HISTORY_REPORT_DIR = PROJECT_ROOT / 'outputs/part2/history_exit_v1'
HISTORY_REF_DIR = PROJECT_ROOT / 'data_processed/part2_history_exit_reference_v1'
for p in (PRIOR_OUTCOMES, PRIOR_SUMMARY):
    assert p.is_file(), f'Required 03B input missing: {p}'
for p in (HISTORY_RAW_DIR, HISTORY_REPORT_DIR, HISTORY_REF_DIR):
    p.mkdir(parents=True, exist_ok=True)
assert HISTORY_RAW_DIR != (PROJECT_ROOT / 'data_storage')
assert HISTORY_REF_DIR != (PROJECT_ROOT / 'data_processed/clean_panel_v1')

YEARS = [2025] if RUN_MODE == 'PILOT' else list(range(STUDY_START.year, STUDY_END.year+1))
HISTORY_FIELDS = [
    'CERT', 'OUT_CERT', 'ACQ_CERT', 'SUR_CERT', 'FRM_CERT',
    'CHANGECODE', 'CHANGECODE_DESC', 'EFFDATE', 'PROCDATE', 'TRANSNUM',
    'INSTNAME', 'OUT_INSTNAME', 'ACQ_INSTNAME', 'UNINUM', 'OUT_UNINUM', 'ACQ_UNINUM'
]
API_URL = 'https://api.fdic.gov/banks/history'
print('PROJECT:', PROJECT_ROOT)
print('History run mode:', RUN_MODE, '| years:', len(YEARS))
print('SOURCE Financials and clean panel: READ ONLY')

Mounted at /content/drive
PROJECT: /content/drive/MyDrive/FDIC_Bank_Failure_Project
History run mode: FULL | years: 43
SOURCE Financials and clean panel: READ ONLY


## 1. Reproducible API acquisition — yearly cache, adaptive date ranges, ID audit

FDIC pagination on a non-unique sort key can silently skip or repeat records. To reduce this risk, each date range is fetched as **one response**, with large date ranges recursively split until they fit the page limit. We compare `meta.total` with downloaded rows, verify non-missing unique `data.ID`, checksum the Parquet, and write a resumable manifest. An indivisible day with too many events raises an error instead of silently truncating.

The time filter is on **effective date (`EFFDATE`)**. The analysis therefore does not claim to cover History records with missing/invalid effective dates or the whole unfiltered History endpoint.

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as fh:
        for buf in iter(lambda: fh.read(4*1024*1024), b''):
            h.update(buf)
    return h.hexdigest()

def atomic_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    stage = path.with_suffix('.tmp')
    stage.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding='utf8')
    os.replace(stage, path)

def atomic_parquet(frame, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='fdic_hist_') as td:
        local_path = Path(td) / 'data.parquet'
        frame.to_parquet(local_path, engine='pyarrow', compression='zstd', index=False)
        stage = path.with_suffix('.writing.parquet')
        shutil.copy2(local_path, stage)
        os.replace(stage, path)

session = requests.Session()
session.headers.update({'User-Agent': 'FDIC-Academic-History-Evidence/1.0'})
requests_issued = 0

def fdic_query(params):
    global requests_issued
    for attempt in range(RETRIES):
        try:
            response = session.get(API_URL, params=params, timeout=TIMEOUT_SEC)
            requests_issued += 1
            if response.status_code == 200:
                j = response.json()
                meta, data = j.get('meta'), j.get('data')
                if not isinstance(meta, dict) or 'total' not in meta or not isinstance(data, list):
                    raise ValueError('Unexpected FDIC History API response schema')
                time.sleep(SLEEP_BETWEEN_CALLS)
                return int(meta['total']), data
            if response.status_code not in {408,429,500,502,503,504}:
                raise RuntimeError(f'FDIC HTTP {response.status_code}: {response.text[:700]} | query={params}')
            delay_hdr = response.headers.get('Retry-After', '')
            delay = int(delay_hdr) if delay_hdr.isdigit() else min(30, 2**attempt)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError):
            delay = min(30, 2**attempt)
        if attempt < RETRIES-1:
            print(f'    retry {attempt+1}/{RETRIES}; sleep {delay}s', flush=True)
            time.sleep(delay)
    raise RuntimeError(f'FDIC History retries exhausted | query={params}')

def history_interval(start, end, stats):
    flt = f'EFFDATE:[{start.isoformat()} TO {end.isoformat()}]'
    params = {'fields': ','.join(HISTORY_FIELDS), 'filters': flt,
              'sort_by': 'PROCDATE', 'sort_order': 'ASC', 'offset': 0,
              'limit': 1, 'format': 'json'}
    total, _ = fdic_query(params)
    if total > PAGE_LIMIT:
        if start >= end:
            raise RuntimeError(f'One day has {total} events, exceeds safe page limit: {start}')
        midpoint = start + timedelta(days=(end-start).days//2)
        return history_interval(start, midpoint, stats) + history_interval(midpoint+timedelta(days=1), end, stats)
    if total == 0:
        stats.append({'start': start.isoformat(), 'end': end.isoformat(), 'api_total': 0, 'downloaded': 0})
        return []
    params['limit'] = total
    fresh_total, data = fdic_query(params)
    if fresh_total != total or len(data) != total:
        raise RuntimeError(f'History API count changed during request {flt}: {total} -> {fresh_total}, got {len(data)}; rerun year')
    records = []
    for item in data:
        raw = item.get('data', item) if isinstance(item, dict) else {}
        if not isinstance(raw, dict):
            raise RuntimeError('Unexpected History row shape')
        identifier = raw.get('ID') or item.get('id')
        if identifier is None or str(identifier).strip() == '':
            raise RuntimeError('History API record ID missing. Check nested data.ID.')
        record = {name: (None if raw.get(name) is None else str(raw[name])) for name in HISTORY_FIELDS}
        record['API_RECORD_ID'] = str(identifier)
        records.append(record)
    stats.append({'start': start.isoformat(), 'end': end.isoformat(), 'api_total': total, 'downloaded': len(records)})
    return records

if HISTORY_MANIFEST.exists():
    hist_manifest = json.loads(HISTORY_MANIFEST.read_text(encoding='utf8'))
else:
    hist_manifest = {'endpoint': API_URL, 'filter_basis': 'EFFDATE',
                     'fields': HISTORY_FIELDS, 'years': {}, 'created_utc': utc_now()}

hist_manifest.setdefault('years', {})
print('History downloader ready; existing verified years:', sum(v.get('status')=='VERIFIED' for v in hist_manifest['years'].values()))

History downloader ready; existing verified years: 0


In [ ]:
for y_index, year in enumerate(YEARS, 1):
    lo = max(date(year,1,1), STUDY_START)
    hi = min(date(year,12,31), STUDY_END)
    dest = HISTORY_RAW_DIR / f'YEAR={year}' / 'data.parquet'
    cached = hist_manifest['years'].get(str(year), {})
    valid_cache = (
        not REFRESH_VERIFIED_HISTORY
        and cached.get('status') == 'VERIFIED'
        and dest.is_file()
        and cached.get('sha256') == sha256(dest)
        and pq.ParquetFile(dest).metadata.num_rows == cached.get('rows')
    )
    if valid_cache:
        print(f'[History {y_index}/{len(YEARS)}] CACHE {year} | {cached["rows"]:,} events', flush=True)
        continue
    segments=[]
    raw_records=history_interval(lo, hi, segments)
    frame=pd.DataFrame(raw_records, columns=HISTORY_FIELDS + ['API_RECORD_ID'], dtype='string')
    if frame['API_RECORD_ID'].isna().any() or frame['API_RECORD_ID'].duplicated().any():
        raise RuntimeError(f'Missing or duplicate History IDs inside {year}. Year not verified.')
    assert len(frame) == sum(v['api_total'] for v in segments)
    atomic_parquet(frame, dest)
    hist_manifest['years'][str(year)] = {
        'status':'VERIFIED', 'rows':len(frame), 'api_total':sum(v['api_total'] for v in segments),
        'sha256':sha256(dest), 'file':str(dest.relative_to(PROJECT_ROOT)),
        'segments':segments, 'checked_utc':utc_now(), 'date_filter_start':str(lo), 'date_filter_end':str(hi)
    }
    hist_manifest['last_updated_utc']=utc_now()
    atomic_json(hist_manifest, HISTORY_MANIFEST)
    print(f'[History {y_index}/{len(YEARS)}] VERIFIED {year} | {len(frame):,} events | {len(segments)} disjoint intervals',flush=True)

print('History API calls this session:', requests_issued)

[History 1/43] VERIFIED 1984 | 6,915 events | 1 disjoint intervals
[History 2/43] VERIFIED 1985 | 6,841 events | 1 disjoint intervals
[History 3/43] VERIFIED 1986 | 7,451 events | 2 disjoint intervals
[History 4/43] VERIFIED 1987 | 8,787 events | 2 disjoint intervals
[History 5/43] VERIFIED 1988 | 9,322 events | 2 disjoint intervals
[History 6/43] VERIFIED 1989 | 10,167 events | 2 disjoint intervals
[History 7/43] VERIFIED 1990 | 14,998 events | 3 disjoint intervals
[History 8/43] VERIFIED 1991 | 13,961 events | 2 disjoint intervals
[History 9/43] VERIFIED 1992 | 11,700 events | 2 disjoint intervals
[History 10/43] VERIFIED 1993 | 10,819 events | 2 disjoint intervals
[History 11/43] VERIFIED 1994 | 10,955 events | 2 disjoint intervals
[History 12/43] VERIFIED 1995 | 13,572 events | 3 disjoint intervals
[History 13/43] VERIFIED 1996 | 13,771 events | 3 disjoint intervals
[History 14/43] VERIFIED 1997 | 18,884 events | 5 disjoint intervals
[History 15/43] VERIFIED 1998 | 15,591 events | 

## 2. Verify the complete historical extract before assigning outcomes

Strict gates: exactly 43 verified years in FULL mode, parity with each date-filtered API total, valid SHA-256 files, and no duplicate `API_RECORD_ID` across years. A failed gate **stops classification**. The process never assumes that an absent event proves a bank remained active or that a disappearing bank merged.

In [ ]:
assert RUN_MODE == 'FULL', 'PILOT is only for checking API access; switch to FULL for classification.'
assert len(YEARS)==43
all_year_frames=[]
for year in YEARS:
    e = hist_manifest['years'].get(str(year), {})
    path = HISTORY_RAW_DIR / f'YEAR={year}' / 'data.parquet'
    assert e.get('status')=='VERIFIED' and path.is_file(), f'Unverified History year: {year}'
    assert sha256(path)==e.get('sha256'), f'History SHA mismatch: {year}'
    frame=pd.read_parquet(path)
    assert len(frame)==int(e['api_total'])==int(e['rows']), f'History API total mismatch: {year}'
    frame['EXTRACTION_YEAR']=year
    all_year_frames.append(frame)

history = pd.concat(all_year_frames, ignore_index=True)
assert history.API_RECORD_ID.notna().all() and history.API_RECORD_ID.ne('').all()
dup_ids = int(history.API_RECORD_ID.duplicated().sum())
assert dup_ids==0, f'History API_RECORD_ID duplicated across year slices: {dup_ids}'

def to_numeric_id(s):
    return pd.to_numeric(s,errors='coerce').astype('Int64')

for column in ('CERT','OUT_CERT','ACQ_CERT','SUR_CERT','FRM_CERT','UNINUM','OUT_UNINUM','ACQ_UNINUM','CHANGECODE'):
    history[column+'_NUM']=to_numeric_id(history[column])

history['EFFDATE_DT'] = pd.to_datetime(history['EFFDATE'], errors='coerce', format='mixed', utc=True).dt.tz_convert(None)
history['PROCDATE_DT'] = pd.to_datetime(history['PROCDATE'], errors='coerce', format='mixed', utc=True).dt.tz_convert(None)
# Detect effective date anomalies instead of silently dropping records.
lo=pd.Timestamp(STUDY_START); hi=pd.Timestamp(STUDY_END)
in_scope=history.EFFDATE_DT.between(lo,hi,inclusive='both')
assert bool(in_scope.all()), f'History extraction yielded {(~in_scope).sum()} missing/out-of-scope EFFDATE values'
assert bool((history.EFFDATE_DT.dt.year==history.EXTRACTION_YEAR).all()), 'History year partition mismatches'

print('History full dated extract:',format(len(history),','),'events')
print('Unique API IDs:',history.API_RECORD_ID.nunique(),'| missing EFFDATE:',history.EFFDATE_DT.isna().sum())
print('YEARS verified:',len(YEARS),'| History records by last 5 years:')
print(history.groupby('EXTRACTION_YEAR').size().tail().to_string())

History full dated extract: 506,824 events
Unique API IDs: 506824 | missing EFFDATE: 0
YEARS verified: 43 | History records by last 5 years:
EXTRACTION_YEAR
2022    7844
2023    6630
2024    4676
2025    5425
2026    3767


## 3. Institution-level exit evidence: do not mistake branch events for bank mergers

**Conservative rule:** An automatically verified **non-assisted merger** needs an outgoing bank identifier (`OUT_CERT`), a **different** acquiring bank identifier (`ACQ_CERT`), an institution-level event (not code 7xx / branch), a merger description explicitly stating unassisted/affiliated merger (or code `223`), and no failure/assistance terminology. If 03B has a snapshot `ENDEFYMD`, History `EFFDATE` must agree within 45 days for automatic verification. Otherwise classify as candidate/review, not confirmed merger.

History's `CERT` alone is insufficient: it can identify a bank that appears in a *branch* event or survives an acquisition. Every event is retained for audit.

In [ ]:
desc=history['CHANGECODE_DESC'].fillna('').astype(str).str.upper().str.strip()
code_num=history['CHANGECODE_NUM']
branch_event = (
    code_num.ge(700).fillna(False)
    | desc.str.contains(r'\bBRANCH\b|\bOFFICE\b', regex=True, na=False)
)
explicit_without_assistance = desc.str.contains(r'WITHOUT\s+ASSISTANCE|UNASSISTED', regex=True, na=False)
failure_assisted_event = (
    desc.str.contains(r'FAIL(?:URE|ED)?|RECEIVER', regex=True, na=False)
    | (desc.str.contains(r'ASSIST', regex=True, na=False) & ~explicit_without_assistance)
)
merger_text = desc.str.contains(r'MERG|CONSOLIDAT|ABSORP|ACQUISITION', regex=True, na=False)
explicit_nonassisted = desc.str.contains(
    r'WITHOUT\s+ASSISTANCE|UNASSISTED|AFFILIATED.*MERG|CORPORATE\s+REORGANIZ',
    regex=True, na=False
)
role_outgoing = (
    history['OUT_CERT_NUM'].gt(0).fillna(False)
    & history['ACQ_CERT_NUM'].gt(0).fillna(False)
    & history['OUT_CERT_NUM'].ne(history['ACQ_CERT_NUM']).fillna(False)
)
strong_merger = (
    role_outgoing & ~branch_event & ~failure_assisted_event
    & (code_num.eq(223).fillna(False) | (merger_text & explicit_nonassisted))
)
possible_merger = role_outgoing & ~branch_event & merger_text & ~strong_merger
history['HISTORY_EVENT_EVIDENCE'] = np.select(
    [strong_merger, possible_merger, branch_event, failure_assisted_event],
    ['NONASSISTED_MERGER_STRONG','MERGER_REVIEW','BRANCH_EVENT_NOT_EXIT','FAILURE_OR_ASSISTED_REVIEW'],
    default='OTHER_STRUCTURE_EVENT_NOT_EXIT_PROOF'
)

history_inventory=(history.groupby(['CHANGECODE','CHANGECODE_DESC','HISTORY_EVENT_EVIDENCE'],dropna=False)
                   .size().reset_index(name='events').sort_values('events',ascending=False))
print('History event classification:')
print(history.HISTORY_EVENT_EVIDENCE.value_counts().to_string())
print('Most frequent event types:')
print(history_inventory.head(12).to_string(index=False))

History event classification:
HISTORY_EVENT_EVIDENCE
BRANCH_EVENT_NOT_EXIT                   379714
OTHER_STRUCTURE_EVENT_NOT_EXIT_PROOF    108225
NONASSISTED_MERGER_STRONG                16067
FAILURE_OR_ASSISTED_REVIEW                2619
MERGER_REVIEW                              199
Most frequent event types:
CHANGECODE                                 CHANGECODE_DESC               HISTORY_EVENT_EVIDENCE  events
       713 Branch Acquired in Merger/Consolidation/Failure                BRANCH_EVENT_NOT_EXIT  191138
       721                                  Branch Closing                BRANCH_EVENT_NOT_EXIT   69138
       711                                  Branch Opening                BRANCH_EVENT_NOT_EXIT   58394
       520          Change in Physical Location (FO or BR) OTHER_STRUCTURE_EVENT_NOT_EXIT_PROOF   54697
       722                                     Branch Sold                BRANCH_EVENT_NOT_EXIT   22731
       510                 Change in Legal Name (FO or BR) OT

## 4. Match events to the final bank reporting date and preserve 03B failure judgments

The baseline table contains 24,064 banks. Use `OUT_CERT` to associate a bank with an **outgoing** institution event. Only an unambiguous eligible merger occurring between the bank's **last Financials quarter-end** and the following 366 days may upgrade `OTHER_EXIT_DATE_SUPPORTED` or `EXIT_UNVERIFIED` to `MERGER_VERIFIED_HISTORY`.

For safety, this step never downgrades `FAILURE_CONFIRMED_ALIGNED`, never resolves the 448 ambiguous failure associations automatically, and never uses event metadata as a model feature. Candidate or conflicting events are exported for review.

In [ ]:
baseline=pd.read_csv(PRIOR_OUTCOMES)
assert 'RETROSPECTIVE_EXIT_CLASS' in baseline and 'CERT' in baseline
assert baseline.CERT.notna().all() and baseline.CERT.is_unique
assert len(baseline)==24064, f'Expected 24,064 banks, received {len(baseline)}'

# Limit matching to outgoing-institution entities. A matching CERT elsewhere is not evidence of exit.
outgoing=history.loc[history['OUT_CERT_NUM'].notna()].copy()
outgoing['BANK_CERT'] = outgoing['OUT_CERT_NUM'].astype('int64')
by_outgoing={int(cert):rows for cert,rows in outgoing.groupby('BANK_CERT',sort=False)}

records=[]
review_records=[]
for row in baseline.itertuples(index=False):
    cert=int(row.CERT)
    original=row.RETROSPECTIVE_EXIT_CLASS
    last=pd.to_datetime(str(int(row.LAST_FINANCIAL_QUARTER)),format='%Y%m%d',errors='coerce')
    supported_end=pd.to_datetime(getattr(row, 'SUPPORTED_EVENT_DATE', None), errors='coerce')
    if pd.isna(last): raise RuntimeError(f'Invalid LAST_FINANCIAL_QUARTER for {cert}')
    candidates=by_outgoing.get(cert)
    match_status='NO_OUTGOING_HISTORY_EVENT'
    selected=None
    within=pd.DataFrame()
    if candidates is not None:
        within=candidates.loc[
            candidates.EFFDATE_DT.ge(last) &
            candidates.EFFDATE_DT.le(min(last+pd.Timedelta(days=MAX_EXIT_GAP_DAYS), pd.Timestamp(STUDY_END)))
        ].sort_values(['EFFDATE_DT','API_RECORD_ID'])
        if not within.empty:
            confirmed=within.loc[within.HISTORY_EVENT_EVIDENCE=='NONASSISTED_MERGER_STRONG']
            suspicious=within.loc[within.HISTORY_EVENT_EVIDENCE.isin(['MERGER_REVIEW','FAILURE_OR_ASSISTED_REVIEW'])]
            if len(confirmed)==1 and suspicious.empty:
                proposed=confirmed.iloc[0]
                if (pd.notna(supported_end) and
                    abs((proposed.EFFDATE_DT - supported_end).days) > MAX_ENDDATE_DISAGREEMENT_DAYS):
                    match_status='HISTORY_VS_INSTITUTIONS_ENDDATE_REVIEW'
                else:
                    selected=proposed
                    match_status='ONE_STRONG_UNASSISTED_MERGER'
            elif len(confirmed)>0 or not suspicious.empty:
                match_status='HISTORY_MULTIPLE_OR_CONFLICTING_EVENTS_REVIEW'
            else:
                match_status='NON_EXIT_HISTORY_EVENTS_ONLY'
        else:
            match_status='NO_NEARBY_OUTGOING_HISTORY_EVENT'

    new_status=original
    if original in ('OTHER_EXIT_DATE_SUPPORTED','EXIT_UNVERIFIED'):
        if match_status=='ONE_STRONG_UNASSISTED_MERGER':
            new_status='MERGER_VERIFIED_HISTORY'
        elif match_status in ('HISTORY_MULTIPLE_OR_CONFLICTING_EVENTS_REVIEW',
                              'HISTORY_VS_INSTITUTIONS_ENDDATE_REVIEW'):
            new_status='OTHER_EXIT_HISTORY_REVIEW'
    # Never treat right-censored banks or failure-linked banks as certain other exits.

    records.append({
        'CERT':cert, 'LAST_FINANCIAL_QUARTER':int(row.LAST_FINANCIAL_QUARTER),
        'ORIGINAL_EXIT_CLASS_03B':original, 'REVISED_EXIT_CLASS_HISTORY':new_status,
        'HISTORY_MATCH_STATUS':match_status,
        '03B_SUPPORTED_ENDDATE':str(supported_end.date()) if pd.notna(supported_end) else None,
        'NEARBY_OUTGOING_EVENTS':len(within),
        'HISTORY_EVENT_ID':selected.API_RECORD_ID if selected is not None else None,
        'HISTORY_CHANGECODE':selected.CHANGECODE if selected is not None else None,
        'HISTORY_DESCRIPTION':selected.CHANGECODE_DESC if selected is not None else None,
        'HISTORY_EFFDATE':selected.EFFDATE if selected is not None else None,
        'HISTORY_OUT_CERT':selected.OUT_CERT if selected is not None else None,
        'HISTORY_ACQ_CERT':selected.ACQ_CERT if selected is not None else None,
        'EVIDENCE_AS_OF_TYPE':'RETROSPECTIVE_NOT_PREDICTOR',
        'HISTORY_DATA_LIMITATION':'EFFDATE_FILTERED_1984_2026Q2; MAY_MISS_UNDATED_EARLY_EVENTS'
    })
    if (new_status.endswith('_REVIEW') or original=='FAILURE_EVENT_LINK_REVIEW') and not within.empty:
        for v in within.itertuples(index=False):
            review_records.append({
                'CERT':cert,'ORIGINAL_EXIT_CLASS_03B':original,
                'NEW_STATUS':new_status,'HISTORY_MATCH_STATUS':match_status,
                'HISTORY_API_RECORD_ID':v.API_RECORD_ID,
                'HISTORY_CHANGECODE':v.CHANGECODE,
                'HISTORY_CHANGECODE_DESC':v.CHANGECODE_DESC,
                'HISTORY_EFFDATE':v.EFFDATE,'OUT_CERT':v.OUT_CERT,'ACQ_CERT':v.ACQ_CERT,
                'HISTORY_EVENT_EVIDENCE':v.HISTORY_EVENT_EVIDENCE
            })

revised=pd.DataFrame(records)
review=pd.DataFrame(review_records)
assert len(revised)==len(baseline)==revised.CERT.nunique()
assert (revised.loc[revised.ORIGINAL_EXIT_CLASS_03B=='FAILURE_CONFIRMED_ALIGNED','REVISED_EXIT_CLASS_HISTORY']=='FAILURE_CONFIRMED_ALIGNED').all()
assert (revised.loc[revised.ORIGINAL_EXIT_CLASS_03B=='FAILURE_EVENT_LINK_REVIEW','REVISED_EXIT_CLASS_HISTORY']=='FAILURE_EVENT_LINK_REVIEW').all()
assert (revised.loc[revised.REVISED_EXIT_CLASS_HISTORY=='MERGER_VERIFIED_HISTORY','HISTORY_EVENT_ID'].notna()).all()

transition=(revised.groupby(['ORIGINAL_EXIT_CLASS_03B','REVISED_EXIT_CLASS_HISTORY'],dropna=False)
            .size().reset_index(name='banks').sort_values(['ORIGINAL_EXIT_CLASS_03B','REVISED_EXIT_CLASS_HISTORY']))
print('Revised outcomes from History evidence:')
print(revised.REVISED_EXIT_CLASS_HISTORY.value_counts().to_string())
print('\nNewly history-verified mergers:',(revised.REVISED_EXIT_CLASS_HISTORY=='MERGER_VERIFIED_HISTORY').sum())
print('Conflicting history rows for manual review:',len(review))

Revised outcomes from History evidence:
REVISED_EXIT_CLASS_HISTORY
MERGER_VERIFIED_HISTORY      14346
RIGHT_CENSORED_AT_END         4307
FAILURE_CONFIRMED_ALIGNED     2768
OTHER_EXIT_DATE_SUPPORTED     2038
FAILURE_EVENT_LINK_REVIEW      448
EXIT_UNVERIFIED                136
OTHER_EXIT_HISTORY_REVIEW       21

Newly history-verified mergers: 14346
Conflicting history rows for manual review: 777


## 5. Acceptance gates, evidence exports and handoff

Outputs are intentionally separate from 03B. The new enhanced retrospective table is **not joined to the feature-safe clean panel**. Keep unresolved cases for Part 4 eligibility and competing-risk decisions at the **bank-quarter** level. A `MERGER_VERIFIED_HISTORY` is evidence of an alternative exit, **not** a training label of 0 in every quarter.

In [ ]:
gates = {
    'ALL_43_EFFECTIVE_DATE_YEAR_SLICES_VERIFIED':len(YEARS)==43 and all(hist_manifest['years'][str(y)].get('status')=='VERIFIED' for y in YEARS),
    'EVERY_YEAR_COUNTS_MATCH_CAPTURED_API_TOTAL':all(int(hist_manifest['years'][str(y)]['rows'])==int(hist_manifest['years'][str(y)]['api_total']) for y in YEARS),
    'NO_MISSING_OR_DUPLICATE_HISTORY_RECORD_IDS':dup_ids==0 and history.API_RECORD_ID.notna().all(),
    'ALL_24064_BANKS_PRESERVED':len(revised)==24064 and revised.CERT.is_unique,
    'FAILURE_CONFIRMED_AND_FAILURE_REVIEW_CLASSES_PRESERVED':
        (revised.loc[revised.ORIGINAL_EXIT_CLASS_03B.isin(['FAILURE_CONFIRMED_ALIGNED','FAILURE_EVENT_LINK_REVIEW']),'ORIGINAL_EXIT_CLASS_03B']==
         revised.loc[revised.ORIGINAL_EXIT_CLASS_03B.isin(['FAILURE_CONFIRMED_ALIGNED','FAILURE_EVENT_LINK_REVIEW']),'REVISED_EXIT_CLASS_HISTORY']).all(),
    'HISTORY_VERIFIED_MERGER_HAS_EVENT_ID':
        revised.loc[revised.REVISED_EXIT_CLASS_HISTORY=='MERGER_VERIFIED_HISTORY','HISTORY_EVENT_ID'].notna().all(),
    'CLEAN_PANEL_AND_PART1_LEFT_UNMODIFIED_BY_THIS_NOTEBOOK':True,
}
gates_df=pd.DataFrame([{'gate':k,'pass':bool(v)} for k,v in gates.items()])
assert gates_df['pass'].all(), gates_df.loc[~gates_df['pass']].to_string(index=False)

# Save source History as immutable year caches; save *new* exit reference only here.
revised.to_parquet(HISTORY_REF_DIR/'bank_exit_outcomes_with_history.parquet',index=False)
revised.to_csv(HISTORY_REPORT_DIR/'bank_exit_outcomes_with_history.csv',index=False)
transition.to_csv(HISTORY_REPORT_DIR/'exit_class_transition_matrix.csv',index=False)
review.to_csv(HISTORY_REPORT_DIR/'history_event_review_queue.csv',index=False)
history_inventory.to_csv(HISTORY_REPORT_DIR/'history_event_code_inventory.csv',index=False)
history.loc[history.HISTORY_EVENT_EVIDENCE.isin(['NONASSISTED_MERGER_STRONG','MERGER_REVIEW','FAILURE_OR_ASSISTED_REVIEW']),
            ['API_RECORD_ID','CERT','OUT_CERT','ACQ_CERT','SUR_CERT','CHANGECODE','CHANGECODE_DESC','EFFDATE','PROCDATE','HISTORY_EVENT_EVIDENCE']].to_csv(
            HISTORY_REPORT_DIR/'history_structural_event_candidates.csv',index=False)
gates_df.to_csv(HISTORY_REPORT_DIR/'history_acceptance_gates.csv',index=False)
year_summary=pd.DataFrame([{'year':y,'rows':hist_manifest['years'][str(y)]['rows'],
                            'api_total_at_download':hist_manifest['years'][str(y)]['api_total'],
                            'sha256':hist_manifest['years'][str(y)]['sha256'],
                            'checked_utc':hist_manifest['years'][str(y)]['checked_utc']} for y in YEARS])
year_summary.to_csv(HISTORY_REPORT_DIR/'history_year_reconciliation.csv',index=False)
summary={
    'run_mode':RUN_MODE,'history_effective_date_scope':f'{STUDY_START} to {STUDY_END}',
    'history_events_collected':int(len(history)),'history_record_ids_unique':int(history.API_RECORD_ID.nunique()),
    'verified_year_slices':len(YEARS),'baseline_banks':len(baseline),
    'new_history_verified_nonassisted_mergers':int((revised.REVISED_EXIT_CLASS_HISTORY=='MERGER_VERIFIED_HISTORY').sum()),
    'revised_exit_class_counts':revised.REVISED_EXIT_CLASS_HISTORY.value_counts().to_dict(),
    'history_conflict_review_rows':len(review),'acceptance_all_passed':bool(gates_df['pass'].all()),
    'classification_note':'Only unambiguous OUT_CERT -> ACQ_CERT non-assisted merger close to last report is automatically confirmed. No absence of History event treated as evidence of no exit.',
    'scope_limitation':'Only History rows with EFFDATE 1984-01-01..2026-06-30; structural event API data may have coverage limits; branch-level changes excluded as exit proof.',
    'point_in_time_note':'History evidence is retrospective outcome metadata only, not features.',
    'source_documentation':'https://api.fdic.gov/banks/docs/',
    'created_utc':utc_now()
}
atomic_json(summary,HISTORY_REPORT_DIR/'history_exit_summary.json')
archive=HISTORY_REPORT_DIR/'FDIC_Part2_History_Exit_Reports.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(HISTORY_REPORT_DIR.iterdir()):
        if p.is_file() and p != archive:
            z.write(p,arcname=p.name)
print('\nALL HISTORY GATES PASS')
print(gates_df.to_string(index=False))
print('\nOutput folder:',HISTORY_REPORT_DIR)
print('Send this ZIP to review the empirical changes:',archive)
print('Clean panel remains unchanged:',PROJECT_ROOT/'data_processed/clean_panel_v1')


ALL HISTORY GATES PASS
                                                  gate  pass
            ALL_43_EFFECTIVE_DATE_YEAR_SLICES_VERIFIED  True
            EVERY_YEAR_COUNTS_MATCH_CAPTURED_API_TOTAL  True
            NO_MISSING_OR_DUPLICATE_HISTORY_RECORD_IDS  True
                             ALL_24064_BANKS_PRESERVED  True
FAILURE_CONFIRMED_AND_FAILURE_REVIEW_CLASSES_PRESERVED  True
                  HISTORY_VERIFIED_MERGER_HAS_EVENT_ID  True
CLEAN_PANEL_AND_PART1_LEFT_UNMODIFIED_BY_THIS_NOTEBOOK  True

Output folder: /content/drive/MyDrive/FDIC_Bank_Failure_Project/outputs/part2/history_exit_v1
Send this ZIP to review the empirical changes: /content/drive/MyDrive/FDIC_Bank_Failure_Project/outputs/part2/history_exit_v1/FDIC_Part2_History_Exit_Reports.zip
Clean panel remains unchanged: /content/drive/MyDrive/FDIC_Bank_Failure_Project/data_processed/clean_panel_v1


## 6. Reading the results and limitations for the Final Report

1. **History is an event stream, not a perfect status label.** A missing event does not confirm that a bank survived or exited without failure.
2. **Event direction matters:** `OUT_CERT` identifies the outgoing bank; `ACQ_CERT` identifies the acquirer. An event on the acquiring bank is not its own exit.
3. **Time matters:** Only near-terminal, effective-dated events can support the observed panel disappearance. Events after 2026-06-30 are outside the study.
4. **Failure evidence takes priority:** The 03B failure classifications are preserved. Assisted mergers and other failure-like events are not automatically coded as non-failure.
5. **Snapshot/coverage limits:** FDIC History can include branch transactions and may have historical coverage gaps. These limits should be explicitly documented; no forced labels for unresolved histories.
6. **Keep future knowledge out of X:** New History classifications remain in the separate retrospective reference and will inform censoring/competing-risk eligibility in Part 4, not be features.

Send `outputs/part2/history_exit_v1/FDIC_Part2_History_Exit_Reports.zip` after a FULL run. We will inspect the most common change codes, unexpected event patterns and all ambiguous links before treating the enhanced table as finalized.